# Classification Model Evaluation

Per-axis evaluation metrics for the I4G fraud classification model.

**Usage:** After a training run completes, update `MODEL_ID` and `GOLDEN_SET_PATH` below,
then run all cells to generate a full evaluation report.

**Phase 0 note:** The stub model returns `UNKNOWN` for all axes, so all metrics will be 0.
This notebook becomes useful once a real model trains in Phase 1.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

from ml.training.baseline import load_golden_set
from ml.training.evaluation import compute_metrics

## Configuration

Set the golden test set path and model prediction path before running.

In [ ]:
# Path to golden test set (JSONL with ground truth labels)
GOLDEN_SET_PATH = "../../data/golden/test.jsonl"  # local copy or gs:// URI

# Path to model predictions (JSONL matching golden set order)
PREDICTIONS_PATH = "../../data/eval/predictions.jsonl"

# Label schema for display
AXES = ["INTENT", "CHANNEL"]

INTENT_LABELS = [
    "INTENT.ROMANCE",
    "INTENT.INVESTMENT",
    "INTENT.IMPERSONATION",
    "INTENT.TECH_SUPPORT",
    "INTENT.SHOPPING",
    "INTENT.EMPLOYMENT",
    "INTENT.OTHER",
]

CHANNEL_LABELS = [
    "CHANNEL.SOCIAL_MEDIA",
    "CHANNEL.EMAIL",
    "CHANNEL.PHONE",
    "CHANNEL.WEBSITE",
    "CHANNEL.MESSAGING",
    "CHANNEL.OTHER",
]

## Load Data

In [ ]:
golden = load_golden_set(GOLDEN_SET_PATH)
print(f"Golden set: {len(golden)} samples")

# Load predictions — each line: {"INTENT": "INTENT.ROMANCE", "CHANNEL": "CHANNEL.EMAIL"}
predictions_raw: list[dict] = []
with open(PREDICTIONS_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            predictions_raw.append(json.loads(line))

print(f"Predictions: {len(predictions_raw)} samples")
assert len(predictions_raw) == len(golden), "Prediction count must match golden set"

## Compute Metrics

In [ ]:
ground_truth = [record["labels"] for record in golden]
result = compute_metrics(predictions_raw, ground_truth)

print(result.summary())

## Per-Axis Metrics Table

In [ ]:
rows = []
for axis_name, m in sorted(result.per_axis.items()):
    rows.append({
        "Axis": axis_name,
        "Precision": round(m.precision, 4),
        "Recall": round(m.recall, 4),
        "F1": round(m.f1, 4),
        "Support": m.support,
    })

# Add overall row
rows.append({
    "Axis": "OVERALL (weighted)",
    "Precision": round(result.overall_precision, 4),
    "Recall": round(result.overall_recall, 4),
    "F1": round(result.overall_f1, 4),
    "Support": result.total_samples,
})

df_metrics = pd.DataFrame(rows)
df_metrics.style.format(
    {"Precision": "{:.4f}", "Recall": "{:.4f}", "F1": "{:.4f}"},
    subset=["Precision", "Recall", "F1"],
)

## Per-Axis F1 Bar Chart

In [ ]:
import matplotlib.pyplot as plt

axis_names = [m.axis for m in result.per_axis.values()]
f1_scores = [m.f1 for m in result.per_axis.values()]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(axis_names, f1_scores, color="steelblue")
ax.set_xlim(0, 1.0)
ax.set_xlabel("F1 Score")
ax.set_title("Per-Axis F1 Scores")
ax.axvline(x=result.overall_f1, color="red", linestyle="--", label=f"Overall F1 = {result.overall_f1:.4f}")
ax.legend()

for bar, score in zip(bars, f1_scores, strict=False):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2, f"{score:.3f}", va="center")

plt.tight_layout()
plt.show()

## Confusion Matrix (per axis)

In [ ]:
from collections import Counter


def plot_confusion_matrix(axis: str, labels: list[str]) -> None:
    """Plot a confusion matrix for a single taxonomy axis."""
    matrix: dict[str, Counter] = {label: Counter() for label in labels}

    for pred, gt in zip(predictions_raw, ground_truth, strict=False):
        true_label = gt.get(axis, "MISSING")
        pred_label = pred.get(axis, "MISSING")
        if true_label in matrix:
            matrix[true_label][pred_label] += 1

    all_labels = labels + ["MISSING"]
    data = [[matrix.get(t, Counter()).get(p, 0) for p in all_labels] for t in all_labels]

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(data, cmap="Blues")
    short = [lbl.split(".")[-1] for lbl in all_labels]
    ax.set_xticks(range(len(all_labels)), short, rotation=45, ha="right")
    ax.set_yticks(range(len(all_labels)), short)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"Confusion Matrix — {axis}")

    for i in range(len(all_labels)):
        for j in range(len(all_labels)):
            val = data[i][j]
            if val > 0:
                ax.text(j, i, str(val), ha="center", va="center", fontsize=9)

    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()


plot_confusion_matrix("INTENT", INTENT_LABELS)
plot_confusion_matrix("CHANNEL", CHANNEL_LABELS)

## Baseline Comparison

Load the saved baseline result and compare against the current model.

In [ ]:
BASELINE_PATH = "../../data/eval/baseline_result.json"  # output of save_baseline_result()

try:
    baseline = json.loads(Path(BASELINE_PATH).read_text())
    baseline_f1 = baseline["overall_f1"]
except FileNotFoundError:
    print("No baseline result found — skipping comparison.")
    baseline = None
    baseline_f1 = None

if baseline_f1 is not None:
    delta = result.overall_f1 - baseline_f1
    print(f"Baseline F1:  {baseline_f1:.4f}")
    print(f"Model F1:     {result.overall_f1:.4f}")
    print(f"Delta:        {delta:+.4f} {'✓ improvement' if delta > 0 else '✗ regression'}")
    print()

    # Per-axis comparison
    comparison_rows = []
    for axis_name, m in sorted(result.per_axis.items()):
        bl_axis = baseline.get("per_axis", {}).get(axis_name, {})
        bl_f1 = bl_axis.get("f1", 0.0)
        comparison_rows.append({
            "Axis": axis_name,
            "Baseline F1": round(bl_f1, 4),
            "Model F1": round(m.f1, 4),
            "Delta": round(m.f1 - bl_f1, 4),
        })

    pd.DataFrame(comparison_rows)

## Summary

This notebook evaluates a trained classification model against the golden test set.

**Next steps for Phase 1:**
- Train a real model (Gemma 2B + LoRA fine-tune) and re-run this notebook
- Add confidence calibration analysis
- Add error analysis (worst predicted samples)
- Integrate with Vertex AI Experiments for tracking across runs